In [1]:
import sys
sys.path.insert(0, 'src')
import train_bidir_buck
from pathlib import Path
from generate_dataset import load_dataset
import wandb

run = wandb.init(
    project="bidir-buck-phnn",
    name="bidir_buck_v1",          # change per experiment
    tags=[ "phnn", "plecs"],
    config={
        "epochs": train_bidir_buck.EPOCHS,
        "lr": train_bidir_buck.LR,
        "multistep_n": train_bidir_buck.MULTISTEP_N,
        "early_stopping_patience": train_bidir_buck.EARLY_STOPPING_PATIENCE,
        "bidir_loss_weight": train_bidir_buck.BIDIR_LOSS_WEIGHT,
    },
)

# Full training configuration
train_bidir_buck.EPOCHS = 12000
train_bidir_buck.EARLY_STOPPING_PATIENCE = 1000  
train_bidir_buck.LR = 5e-4

trajectories = load_dataset('data/plecs_bidir_train1.npz')
print(f"Loaded {len(trajectories)} trajectories")

train_idx, val_idx = train_bidir_buck.split_by_trajectory(
    len(trajectories), train_bidir_buck.SEED)
print(f"Train: {len(train_idx)}  Val: {len(val_idx)}")

ckpt = train_bidir_buck.train(
    trajectories,
    train_idx,
    val_idx,
    model_path=Path('models/bidir_buck_phnn.pt'),
    loss_plot_path=Path('tests/bidir_buck_loss.png'),
    traj_plot_path=Path('tests/bidir_buck_traj.png'),
    wandb_run = run,
)

wandb.finish()

print(f"\nTraining complete.")
print(f"Best epoch: {ckpt['best_epoch']}")
print(f"Best val loss: {ckpt['best_val_loss']:.6e}")
p = ckpt['learned_params']
print(f"R11={p.get('R11', 'n/a'):.5f}  R12={p.get('R12', 'n/a'):.5f}  R22={p.get('R22', 'n/a'):.5f}")
print(f"delta_G_norm={p.get('delta_G_norm', 'n/a'):.4f}")
print(f"H_NN_norm={p.get('H_NN_norm', 'n/a'):.4f}")

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\rcper\_netrc.
wandb: Currently logged in as: rcpersico2 (rcpersico2-ucla) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Loaded 30 trajectories
Train: 24  Val: 6
Device              : cuda
Trajectories total  : 30
Train windows       : 1064 x 60 steps  (bidirectional: 228, unidirectional: 836)
Val   windows       : 239 x 60 steps  (bidirectional: 48, unidirectional: 191)
Training dt         : 50.0 µs
Dataset dt median   : 50.0 µs
State scale [A, V]  : [0.9500971 2.5514202]

 Epoch         train           val        lr       R11       R22      |H|uni       |H|bi     |dG|uni      |dG|bi
----------------------------------------------------------------------------------------------------------------
     1  1.707603e-01  2.059555e+00  5.00e-04   0.10403   0.04723  6.6471e-04  7.0465e-04  8.1524e-03  8.1671e-03
   100  4.174785e-02  6.064939e-02  5.00e-04   0.22688   0.09950  3.9060e-03  3.8523e-03  7.8354e-02  8.0804e-02
   200  2.926650e-02  1.244430e-01  5.00e-04   0.22295   0.15319  3.5369e-03  3.4930e-03  1.9504e-01  2.1486e-01
   300  2.456285e-02  1.104156e-01  2.50e-04   0.21030   0.16559  3.2840e-03 

wandb: WARNING Linked 1 file into the W&B run directory (hardlinks); call wandb.save again to sync new files.



Early stopping at epoch 1609 (best val=4.975360e-02 at epoch 609).

Best epoch  : 609   best val loss : 4.975360e-02
R_theta at rep. pt: R11=0.18717  R12=-0.17253  R22=0.16114
delta_G_norm=2.9521e-01  H_NN_norm=4.1839e-03
Saved model : models\bidir_buck_phnn.pt
Saved loss plot : tests\bidir_buck_loss.png


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


H_NN_norm,▁▇▇▆▇███▇▇▇▇▆▆▆▆▆
H_NN_norm_bidir,▁█▇▆▇███▇▇▇▆▆▆▆▆▆
H_NN_norm_unidir,▁▇▇▆▇███▇▇▇▇▇▇▇▆▆
R11,▁██▇▇▆▆▅▄▄▄▃▃▃▃▃▃
R12,█▄▁▁▁▁▁▁▂▂▂▂▂▂▂▂▂
R22,▁▄▇██████████████
dG_norm_bidir,▁▂▃▄▄▅▅▆▇▇▇██████
dG_norm_unidir,▁▂▃▄▄▅▅▆▇▇▇██████
delta_G_norm,▁▂▃▄▄▅▅▆▇▇▇▇█████
lr,███▄▄▄▄▄▄▃▃▂▂▁▁▁▁
+3,...



Training complete.
Best epoch: 609
Best val loss: 4.975360e-02


ValueError: Unknown format code 'f' for object of type 'str'

In [2]:
import sys
sys.path.insert(0, 'src')
import numpy as np
import torch
import matplotlib.pyplot as plt
from pathlib import Path
from generate_dataset import load_dataset
from phnn_bidir import BidirBuckPHNN

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load trained model
ckpt = torch.load('models/bidir_buck_phnn.pt', map_location='cpu', weights_only=False)
model = BidirBuckPHNN(L=100e-6, C=100e-6,
                    initial_R_L=0.10, initial_R_C=0.05).to(device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

train_losses = ckpt['train_losses']
val_losses   = ckpt['val_losses']
best_epoch   = ckpt['best_epoch']

fig, ax = plt.subplots(figsize=(10, 4))
epochs = range(1, len(train_losses) + 1)
ax.semilogy(epochs, train_losses, label='train', alpha=0.8)
ax.semilogy(epochs, val_losses,   label='validation', alpha=0.8)
ax.axvline(best_epoch, color='gray', ls='--', alpha=0.6,
           label=f'best (epoch {best_epoch})')
ax.set_xlabel('Epoch')
ax.set_ylabel('Normalised 60-step rollout MSE')
ax.set_title('Bidir Buck PHNN — Training Loss')
ax.legend(); ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.savefig('tests/bidir_loss_curves.png', dpi=150)
plt.show()
print(f"Best val loss: {ckpt['best_val_loss']:.6e}  at epoch {best_epoch}")

def rk4_rollout(model, x0, t, u, device):
    states = np.zeros((len(t), 2), dtype=np.float32)
    states[0] = x0
    for k in range(len(t) - 1):
        dt_k = float(t[k+1] - t[k])
        def f(x_np):
            xt = torch.tensor(x_np[None], dtype=torch.float32, device=device)
            ut = torch.tensor(u[k:k+1],  dtype=torch.float32, device=device)
            with torch.no_grad():
                return model(xt, ut).cpu().numpy()[0]
        k1 = f(states[k])
        k2 = f(states[k] + 0.5*dt_k*k1)
        k3 = f(states[k] + 0.5*dt_k*k2)
        k4 = f(states[k] +     dt_k*k3)
        states[k+1] = states[k] + (dt_k/6)*(k1 + 2*k2 + 2*k3 + k4)
    return states

trajs = load_dataset('data/plecs_bidir_test1.npz')  # held-out test set

# Compute regime on the fly, pick 2 unidirectional and 2 bidirectional trajectories
annotated = []
for i, td in enumerate(trajs):
    pct_neg = float(np.mean(td['i_L'] < -0.05))
    annotated.append((i, td, pct_neg))

unidir_trajs = sorted([x for x in annotated if x[2] < 0.05],
                   key=lambda x: x[2])
bidir_trajs = sorted([x for x in annotated if x[2] >= 0.05],
                   key=lambda x: x[2], reverse=True)

selected = []
if len(unidir_trajs) >= 2:
    selected += unidir_trajs[:2]
elif len(unidir_trajs) == 1:
    selected += unidir_trajs[:1]
if len(bidir_trajs) >= 2:
    selected += bidir_trajs[:2]
elif len(bidir_trajs) == 1:
    selected += bidir_trajs[:1]

print(f"\nSelected {len(selected)} trajectories for evaluation:")
for idx, td, regime in selected:
    print(f"  traj {idx}: regime={regime*100:.1f}% reverse i_L  "
          f"alpha={td['alpha'].min():.2f}  “{td['alpha'].max():.2f}  "
          f"V_in={td['V_in'][0]:.1f}V")

n = len(selected)
fig, axes = plt.subplots(n, 2, figsize=(14, 3.5*n))
if n == 1:
    axes = axes[np.newaxis, :]

rmse_iL_all, rmse_VC_all = [], []

for row, (idx, td, regime) in enumerate(selected):
    x0  = np.array([td['i_L'][0], td['V_C'][0]], dtype=np.float32)
    u   = np.stack([td['alpha']*td['V_in'], td['i_o']], axis=1).astype(np.float32)
    pred = rk4_rollout(model, x0, td['t'], u, device)

    rmse_iL = np.sqrt(np.mean((pred[:,0] - td['i_L'])**2))
    rmse_VC = np.sqrt(np.mean((pred[:,1] - td['V_C'])**2))
    rmse_iL_all.append(rmse_iL)
    rmse_VC_all.append(rmse_VC)

    t_ms  = td['t'] * 1e3
    label = ('Unidir' if regime < 0.05 else 'Bidir')
    title = (f"Traj {idx} [{label}]  regime={regime*100:.1f}%  "
             f"Î±={td['alpha'].min():.2f}  “{td['alpha'].max():.2f}  "
             f"V_in={td['V_in'][0]:.1f}V")

    axes[row,0].plot(t_ms, td['i_L'],  'k--', lw=1.5, label='PLECS')
    axes[row,0].plot(t_ms, pred[:,0],  'b-',  lw=1.5, label='Bidir PHNN')
    axes[row,0].axhline(0, color='gray', lw=0.5, alpha=0.4, ls=':')
    axes[row,0].set_ylabel('i_L [A]')
    axes[row,0].set_title(title)
    axes[row,0].legend(fontsize=8); axes[row,0].grid(True, alpha=0.3)
    axes[row,0].text(0.98, 0.05, f'RMSE={rmse_iL:.3f}A',
                     transform=axes[row,0].transAxes,
                     ha='right', fontsize=8, color='blue')

    axes[row,1].plot(t_ms, td['V_C'],  'k--', lw=1.5, label='PLECS')
    axes[row,1].plot(t_ms, pred[:,1],  'r-',  lw=1.5, label='Bidir PHNN')
    axes[row,1].set_ylabel('V_C [V]')
    axes[row,1].set_title(title)
    axes[row,1].legend(fontsize=8); axes[row,1].grid(True, alpha=0.3)
    axes[row,1].text(0.98, 0.05, f'RMSE={rmse_VC:.3f}V',
                     transform=axes[row,1].transAxes,
                     ha='right', fontsize=8, color='red')

for ax in axes[-1,:]:
    ax.set_xlabel('Time [ms]')

fig.suptitle('Bidir Buck PHNN — Test Set Trajectories (held-out)', fontsize=12)
fig.tight_layout()
plt.savefig('tests/bidir_test_trajectories.png', dpi=150)
plt.show()

print(f"\n{'='*50}")
print(f"SUMMARY — Bidir PHNN on held-out test set")
print(f"{'='*50}")
unidir_sel  = [(rmse_iL_all[i], rmse_VC_all[i])
            for i, (_, _, r) in enumerate(selected) if r < 0.05]
bidir_sel  = [(rmse_iL_all[i], rmse_VC_all[i])
            for i, (_, _, r) in enumerate(selected) if r >= 0.05]

if unidir_sel:
    print(f"Unidirectional trajectories ({len(unidir_sel)}):")
    print(f"  Mean RMSE i_L: {np.mean([x[0] for x in unidir_sel]):.4f} A")
    print(f"  Mean RMSE V_C: {np.mean([x[1] for x in unidir_sel]):.4f} V")
if bidir_sel:
    print(f"Bidirectional trajectories ({len(bidir_sel)}):")
    print(f"  Mean RMSE i_L: {np.mean([x[0] for x in bidir_sel]):.4f} A")
    print(f"  Mean RMSE V_C: {np.mean([x[1] for x in bidir_sel]):.4f} V")

p = ckpt['learned_params']
print(f"\nLearned parameters:")
print(f"  R11 (inductor branch): {p.get('R11', 'n/a')}")
print(f"  R22 (capacitor branch): {p.get('R22', 'n/a')}")
print(f"  delta_G_norm: {p.get('delta_G_norm', 'n/a'):.4f}")
print(f"  H_NN_norm:    {p.get('H_NN_norm', 'n/a'):.4f}")
print(f"{'='*50}")

C:\Users\rcper\AppData\Local\Temp\ipykernel_43212\4027855793.py:35: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Best val loss: 4.975360e-02  at epoch 609

Selected 3 trajectories for evaluation:
  traj 1: regime=2.4% reverse i_L  alpha=0.26â€“0.68  V_in=9.9V
  traj 2: regime=3.7% reverse i_L  alpha=0.47â€“0.72  V_in=13.0V
  traj 3: regime=5.0% reverse i_L  alpha=0.22â€“0.60  V_in=8.1V

SUMMARY — Bidir PHNN on held-out test set
Unidirectional trajectories (2):
  Mean RMSE i_L: 0.1421 A
  Mean RMSE V_C: 0.2429 V
Bidirectional trajectories (1):
  Mean RMSE i_L: 0.0851 A
  Mean RMSE V_C: 0.1413 V

Learned parameters:
  R11 (inductor branch): n/a
  R22 (capacitor branch): n/a
  delta_G_norm: 0.2952
  H_NN_norm:    0.0042


C:\Users\rcper\AppData\Local\Temp\ipykernel_43212\4027855793.py:132: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
